# Tutorial: Generalized Contact Matrices with `cntmosaic`

**Generalized contact matrices** capture how people mix not just by age, but by additional demographic features such as gender, socioeconomic status, or race/ethnicity. They are key inputs for estimating disease reproduction numbers and designing interventions that account for social inequalities in contact behaviour.

This tutorial walks through a complete `cntmosaic` workflow using data from the UK branch of the **POLYMOD study**, stratifying contact patterns by **age and gender**. We cover two scenarios:

- **Part 1 — Complete data:** Gender is observed for both respondents *and* their contacts. We fit the `GenMixCC` model and visualize the estimated contact intensity matrices.
- **Part 2 — Partial data:** Gender is observed only for respondents. We fit a partially-stratified model, then use *feasible mixing bounds* and a truncated Dirichlet sampler to predict the fully-stratified matrices.

**Workflow at a glance:**
1. Load and preprocess the survey data
2. Initialize data containers
3. Specify and fit the model (complete-data setting)
4. Visualize the estimated contact intensity matrices
5. Re-fit under the partial-data setting (gender of contacts unobserved)
6. Predict fully-stratified matrices from the partial estimates

In [ ]:
import numpy as np
import pandas as pd

import jax

# Enable 64-bit precision in JAX for better numerical stability
jax.config.update("jax_enable_x64", True)

from jax.random import PRNGKey
from numpyro.infer.autoguide import AutoNormal

from cntmosaic.dataloader import (
    ContactSurveyLoader,
    ParticipantData,
    ContactData,
    PopulationData,
    StratificationData,
)
from cntmosaic.utils import AgeGroupSpecs
from cntmosaic.models import GenMixCC
from cntmosaic.models.numpyro.priors import vdKassteele2D
from cntmosaic.analysis import ModelSummariser, ContactSummary
from cntmosaic.predict import Predictor
from cntmosaic.vis import plot_mosaic_pixilated

import altair as alt

---
## Part 1: Complete-Data Setting

### Step 1: Load the Data

We use data from the UK branch of the **POLYMOD study** (Mossong et al., 2008), a landmark European contact survey conducted across 8 countries covering roughly 7,000 participants. It remains one of the most widely used contact datasets in infectious disease modelling. The original data are publicly available at [socialcontactdata.org](https://www.socialcontactdata.org).

We prepare three pieces of data:
1. **Participant data** — age and gender of each survey respondent
2. **Contact data** — individual contact records with the reported age and gender of each contact
3. **Population data** — population size estimates by age and sex (2011 UK Census)

Ages are binned into **5-year age groups** (e.g., [0, 5), [5, 10), …, [80, 85)). Many public contact datasets only provide coarse-age information for contacts, so coarse-age models are more broadly applicable.

**Note:** Some contact records will be dropped for missing age or gender values — this is expected in real-world survey data and is handled automatically by the data containers in the next step.

In [ ]:
# ===== Participant data ======
# Load
df_part = pd.read_csv("data/POLYMOD_UK_participant_common.csv")

# Preprocess
df_part["part_age_grp"] = pd.cut(df_part["part_age"], bins=range(0, 86, 5), right=False)
df_part["part_gender"] = pd.Categorical(df_part["part_gender"], categories=["M", "F"])

# ===== Contact data =====
# Load
df_cnt = pd.read_csv("data/POLYMOD_UK_contact_common.csv")

# Preprocess
df_cnt["cnt_age_grp"] = pd.cut(
    df_cnt["cnt_age_exact"], bins=range(0, 86, 5), right=False
)
df_cnt["cnt_gender"] = pd.Categorical(df_cnt["cnt_gender"], categories=["M", "F"])

# ===== Population data =====
# Load
df_pop = pd.read_csv("data/UK_population_2011.csv")

# Preprocess
df_pop["age"] = df_pop["age"].str.replace("+", "").astype(int)
df_pop = df_pop[df_pop["age"] <= 84].copy()
df_pop = df_pop.rename(columns={"Male": "M", "Female": "F"}).melt(
    id_vars="age", var_name="gender", value_name="P"
)
df_pop["age_grp"] = pd.cut(df_pop["age"], bins=range(0, 86, 5), right=False)
df_pop = df_pop.groupby(["age_grp", "gender"], observed=True)["P"].sum().reset_index()
df_pop["gender"] = pd.Categorical(df_pop["gender"], categories=["M", "F"])

In [ ]:
print("Participant data:")
display(
    df_part[
        ["part_id", "part_age", "part_age_grp", "part_gender", "part_gender"]
    ].head()
)
print(f"\n{df_part.shape[0]} participants total\n")

print("Contact data:")
display(
    df_cnt[
        ["part_id", "cnt_age_exact", "cnt_age_grp", "cnt_gender", "cnt_gender"]
    ].head()
)
print(f"\n{df_cnt.shape[0]} contact records total (before dropping missing values)")

### Step 2: Initialize Data Containers

The raw DataFrames need to be converted into typed containers that the model classes can consume. `cntmosaic` provides four container classes:

| Container            | Purpose                                                   |
|----------------------|-----------------------------------------------------------|
| `ParticipantData`    | Participant age and stratification features               |
| `ContactData`        | Contact records with age and stratification features      |
| `PopulationData`     | Population sizes by age group                             |
| `StratificationData` | Stratification weights (proportions or counts by stratum) |

Each container is initialized with the DataFrame and the names of the relevant columns. This tells downstream classes which columns hold age and stratification information, and runs basic validation to catch common data issues early.

**Note on `StratificationData`:** We use the `.from_counts()` constructor below, which is convenient when stratified population counts are available. If you only have proportions, use `.from_proportions()` instead.

In [ ]:
part_data = ParticipantData(
    data=df_part,
    id_col="part_id",
    age_grp_col="part_age_grp",
    strat_var_cols="part_gender",
)

cnt_data = ContactData(
    data=df_cnt,
    id_col="part_id",
    age_grp_col="cnt_age_grp",
    strat_var_cols="cnt_gender",
)

pop_data = PopulationData(
    data=df_pop,
    age_grp_col="age_grp",
    size_col="P",
    strat_var_cols="gender",
)

strat_data = StratificationData.from_counts(
    data=df_pop, age_grp_col="age_grp", strat_var_cols="gender", count_col="P"
)

The four containers are then passed to `ContactSurveyLoader`, which consolidates and processes them into the internal format expected by the model classes.

In [ ]:
dataloader = ContactSurveyLoader.from_containers(
    part_data=part_data, cnt_data=cnt_data, pop_data=pop_data, strat_data=strat_data
)

### Step 3: Specify the Model

We use `GenMixCC` — **Gen**eralized **Mix**ing with **C**oarse-age participants and **C**oarse-age contacts. This is the right model class when both respondents and contacts are grouped into age ranges rather than 1-year bands.

The model requires a prior for each component:
- **`"rate"`** — the baseline age-by-age contact rate, shared across all strata
- **`"gender"`** — the contact rate modifier for the gender stratification

We use `vdKassteele2D`, a 2-D Gaussian Markov Random Field (GMRF) prior. It borrows strength across neighboring age groups and enforces **age reciprocity** — symmetric contact rates between age pairs — by construction.

The `prior_type` argument controls which data setting the prior is configured for:
- `"global"` — always used for the baseline rate
- `"full"` — for a stratification feature when gender is observed for **both** respondents and contacts (this part)
- `"partial"` — for a stratification feature when gender is observed for **respondents only** (Part 2)

**Note on `AgeGroupSpecs`:** The optional `age_group_specs` argument tells the model how ages are binned. It is not required for fitting but is used by the visualization functions to label axes correctly.

In [ ]:
age_grp_specs = AgeGroupSpecs(0, 84, 5)

# Specify priors
priors = {
    "rate": vdKassteele2D(prior_type="global"),
    "gender": vdKassteele2D(prior_type="full"),
}

# Initialize the model
model = GenMixCC(
    dataloader=dataloader,
    priors=priors,
    likelihood="negbin",
    age_group_specs=age_grp_specs,  # Optional
)

# Print the dimensions of the model's parameters to verify they match the expected shapes
# Also serves as a quick check before running inference
model.print_model_shape()

### Step 4: Run Inference

`cntmosaic` supports two inference backends: **stochastic variational inference (SVI)** and **MCMC**, both via NumPyro. SVI is much faster and is sufficient for most analyses. Here we use mean-field variational inference with the `AutoNormal` guide from NumPyro.

In [ ]:
prng_key = PRNGKey(1)
guide = AutoNormal(model.model)
# 10k steps is sufficient for this tutorial; use 20k for production analyses
model.run_inference_svi(prng_key=prng_key, guide=guide, num_steps=10_000)

### Step 5: Summarize and Visualize Contact Intensities

Use `ModelSummariser` to extract posterior summaries of the estimated contact intensities. `summarise_cint()` returns a dictionary of `ContactSummary` objects — one per stratum pair — each holding the posterior median and 95% credible interval bounds.

For a model with two gender strata (M, F), the dictionary contains four entries: `"M->M"`, `"M->F"`, `"F->M"`, and `"F->F"`.

In [ ]:
summarizer = ModelSummariser(model)
summary_cint = summarizer.summarise_cint(alpha=0.95, measure="median")

`cntmosaic` uses `altair` for plotting. `plot_mosaic_pixilated()` renders a heatmap for coarse-age estimates. Passing a `ContactSummary` object directly plots the posterior median contact intensity for each age-pair cell.

In [ ]:
chart_kwargs = {
    "xlabel": "Age of respondent",
    "ylabel": "Age of contact",
    "zlabel": "Intensity",
    "color_min": 0,
    "color_max": 5.0,
}

chart_mm = plot_mosaic_pixilated(
    summary_cint["M->M"],
    title="Male to Male",
    show_x_label=False,
    show_x_ticks=False,
    **chart_kwargs,
)
chart_mf = plot_mosaic_pixilated(
    summary_cint["M->F"],
    title="Male to Female",
    show_y_label=False,
    show_y_ticks=False,
    show_x_label=False,
    show_x_ticks=False,
    **chart_kwargs,
)
chart_fm = plot_mosaic_pixilated(
    summary_cint["F->M"],
    title="Female to Male",
    **chart_kwargs,
)
chart_ff = plot_mosaic_pixilated(
    summary_cint["F->F"],
    title="Female to Female",
    show_y_label=False,
    show_y_ticks=False,
    **chart_kwargs,
)

top_row = alt.hconcat(chart_mm, chart_mf, spacing=25)
bottom_row = alt.hconcat(chart_fm, chart_ff, spacing=17)
final_chart = alt.vconcat(top_row, bottom_row, spacing=25)

final_chart

**Interpreting the results:**
- **Strong diagonal** — assortative mixing: people tend to contact others of similar age.
- **Off-diagonal block** — parent–child contacts, visible as a horizontal and vertical band around younger ages.
- **M→F and F→M symmetry** — these two matrices are near-transposes of each other, reflecting the reciprocity constraint built into the model.

The `ContactSummary.lower` and `.upper` attributes hold the 95% credible interval bounds for each cell, which can be used to quantify estimation uncertainty.

---
## Part 2: Partial-Data Setting

In practice, survey respondents can reliably report their own demographic features (gender, income, socioeconomic status), but they rarely know the same information for their contacts. We call this the **partial-data setting**.

In this part we re-run the analysis *pretending* that gender information for contacts was not collected. This mirrors real-world situations for features like income or SES.

**Only two things change from Part 1:**
1. `ContactData` is initialized without `strat_var_cols` — no contact-side gender column
2. The prior for `"gender"` switches from `prior_type="full"` to `prior_type="partial"`

Everything else — data loading, model class, inference procedure — stays the same.

In [ ]:
cnt_data = ContactData(
    df_cnt, id_col="part_id", age_grp_col="cnt_age_grp"
)  # ← changed: no strat_var_cols
pop_data = PopulationData(df_pop, age_grp_col="age_grp", size_col="P")

In [ ]:
dataloader = ContactSurveyLoader.from_containers(
    part_data=part_data, cnt_data=cnt_data, pop_data=pop_data, strat_data=strat_data
)

We also update the prior for `"gender"`: switching from `"full"` to `"partial"` reduces the parameter space to match the partial stratification, where only respondent-side gender effects are estimated directly.

In [ ]:
age_grp_specs = AgeGroupSpecs(0, 84, 5)

priors = {
    "rate": vdKassteele2D(prior_type="global"),
    "gender": vdKassteele2D(prior_type="partial"),  # ← changed: partial
}

model = GenMixCC(
    dataloader=dataloader,
    priors=priors,
    likelihood="negbin",
    age_group_specs=age_grp_specs,
)

model.print_model_shape()

In [ ]:
prng_key = PRNGKey(1)
guide = AutoNormal(model.model)
model.run_inference_svi(prng_key=prng_key, guide=guide, num_steps=10_000)

In [ ]:
summarizer = ModelSummariser(model)
summary_cint = summarizer.summarise_cint(alpha=0.95, measure="median")

In [ ]:
chart_kwargs = {
    "xlabel": "Age of respondent",
    "ylabel": "Age of contact",
    "zlabel": "Intensity",
    "color_min": 0,
    "color_max": 7.0,
}

chart_m = plot_mosaic_pixilated(
    summary_cint["M->All"],
    title="Male to All",
    **chart_kwargs,
)
chart_f = plot_mosaic_pixilated(
    summary_cint["F->All"],
    title="Female to All",
    show_y_label=False,
    show_y_ticks=False,
    **chart_kwargs,
)

final_chart = alt.hconcat(chart_m, chart_f)
final_chart

### Predicting Fully-Stratified Matrices

With the partial model fitted, we can predict the fully-stratified contact matrices using `Predictor`. The prediction relies on **feasible mixing bounds**:

1. The partial model estimates how many contacts each gender group makes in total (the row and column margins).
2. Those margins constrain the inner cells: knowing that Male respondents report X total contacts places a lower bound on Male→Male and an upper bound on Male→Female contacts.
3. `Predictor` samples the attributable fractions η (the proportion of contacts directed from respondent group k to contact group ℓ) from a truncated Dirichlet prior that respects these bounds, then converts them to fully-stratified contact intensities.

Predictions will be **wider and more uncertain** than the Part 1 estimates, because the gender breakdown among contacts is inferred indirectly from the margins alone.

In [ ]:
predictor = Predictor(summarizer, dataloader)
predictions = predictor.predict_full_matrices(rng=np.random.default_rng(1))

In [ ]:
# Predictor returns raw samples (shape: [n_samples, A, A] per stratum pair).
# We summarize them into ContactSummary objects so they can be passed directly
# to plot_mosaic_pixilated(), the same as in Part 1.
prediction_summaries = {}
for key, pred in predictions.items():
    lb, center, ub = np.quantile(pred, [0.025, 0.5, 0.975], axis=0)
    cs = ContactSummary(
        lower=lb,
        central=center,
        upper=ub,
        alpha=0.95,
        measure="median",
        age_group_specs=age_grp_specs,
    )
    prediction_summaries[key] = cs

In [ ]:
chart_kwargs = {
    "xlabel": "Age of respondent",
    "ylabel": "Age of contact",
    "zlabel": "Intensity",
    "color_min": 0,
    "color_max": 5.0,
}

chart_mm = plot_mosaic_pixilated(
    prediction_summaries["M->M"],
    title="Male to Male",
    show_x_label=False,
    show_x_ticks=False,
    **chart_kwargs,
)
chart_mf = plot_mosaic_pixilated(
    prediction_summaries["M->F"],
    title="Male to Female",
    show_y_label=False,
    show_y_ticks=False,
    show_x_label=False,
    show_x_ticks=False,
    **chart_kwargs,
)
chart_fm = plot_mosaic_pixilated(
    prediction_summaries["F->M"],
    title="Female to Male",
    **chart_kwargs,
)
chart_ff = plot_mosaic_pixilated(
    prediction_summaries["F->F"],
    title="Female to Female",
    show_y_label=False,
    show_y_ticks=False,
    **chart_kwargs,
)

top_row = alt.hconcat(chart_mm, chart_mf, spacing=25)
bottom_row = alt.hconcat(chart_fm, chart_ff, spacing=17)
final_chart = alt.vconcat(top_row, bottom_row, spacing=25)

final_chart

**Interpreting the predictions:**

Compared to the fully-stratified estimates from Part 1, these predicted matrices are wider (more uncertain) because the gender breakdown among contacts was never observed — it is recovered indirectly from the row and column totals alone. The central estimates should nonetheless be broadly consistent with the Part 1 results.

---
## Summary

In this tutorial you:
- Loaded and preprocessed POLYMOD UK contact survey data
- Organized the data into `cntmosaic` containers
- Fitted `GenMixCC` in both the **complete-data** (Part 1) and **partial-data** (Part 2) settings
- Visualized posterior contact intensity matrices
- Predicted fully-stratified matrices from a partially-stratified fit using feasible mixing bounds